<a href="https://colab.research.google.com/github/marry12256/urdu-ocr-codesaviours-si26-maryam/blob/main/SI26_Week2_maryam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Urdu OCR Image Preprocessing

This notebook preprocesses Urdu OCR images by converting them to grayscale, resizing them, removing noise, and applying thresholding. The processed images will be used for OCR testing.

In [ ]:
!pip install opencv-python-headless pillow

In [ ]:
import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
def preprocess_image(image_path, save_path):

    img = cv2.imread(image_path)

    if img is None:
        print(f"Could not load: {image_path}")
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    resized = cv2.resize(gray, (512, 128))

    denoised = cv2.fastNlMeansDenoising(resized, h=10)

    _, binary = cv2.threshold(denoised, 127, 255, cv2.THRESH_BINARY)

    cv2.imwrite(save_path, binary)

    return binary

os.makedirs("data/processed", exist_ok=True)

print("Preprocessing function ready!")

Preprocessing function ready!


In [ ]:
import os

input_folder = "/content/drive/MyDrive/images"
output_folder = "/content/drive/MyDrive/processed_images"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        image_path = os.path.join(input_folder, filename)
        save_path = os.path.join(output_folder, filename)

        preprocess_image(image_path, save_path)

print("✅ All images have been preprocessed successfully!")

✅ All images have been preprocessed successfully!


In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-urd
!pip install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-urd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 1,000 kB of archives.
After this operation, 1,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-urd all 1:4.00~git30-7274cfa-1.1 [1,000 kB]
Fetched 1,000 kB in 1s (669 kB/s)
Selecting previously unselected package tesseract-ocr-urd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-urd_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-urd (1:4.00~git30-7274cfa-1.1) ...


In [ ]:
import pytesseract
from PIL import Image
import glob

In [ ]:
import glob
from PIL import Image
import pytesseract

test_images = glob.glob("/content/drive/MyDrive/processed_images/*.jpg")
test_images += glob.glob("/content/drive/MyDrive/processed_images/*.jpeg")
test_images += glob.glob("/content/drive/MyDrive/processed_images/*.png")

test_images = test_images[:5]

print("=== Tesseract Results on Urdu Images ===\n")

for img_path in test_images:
    img = Image.open(img_path)

    result = pytesseract.image_to_string(
        img,
        lang="urd",
        config="--oem 3 --psm 6"
    )

    print(f"Image: {img_path}")
    print("OCR Output:")
    print(result)
    print("-" * 50)

=== Tesseract Results on Urdu Images ===

Image: /content/drive/MyDrive/processed_images/sign3.jpg.jpeg
OCR Output:
‎٠‏ ٹا
ہس تی ۴سط ہرد سور ری

--------------------------------------------------
Image: /content/drive/MyDrive/processed_images/book1.jpg.jpeg
OCR Output:

--------------------------------------------------
Image: /content/drive/MyDrive/processed_images/news1.jpg.jpeg
OCR Output:
غیرملکی خیرزسال الارے اے اہی کی رپوٹ کے

--------------------------------------------------
Image: /content/drive/MyDrive/processed_images/news11.jpg.jpeg
OCR Output:
9۰۶

--------------------------------------------------
Image: /content/drive/MyDrive/processed_images/news10.jpg.jpeg
OCR Output:
وپ اہ اک ار ری کے لوان

--------------------------------------------------


## Gap Analysis
### Image 1: sign3.jpg.jpeg

**Actual Urdu Text:**

آگے پٹرول پمپ ہے۔

**Tesseract Output:**
 ٹا
ہس تی ۴سط ہرد سور ری


**What went wrong?**

Tesseract incorrectly recognized the Urdu text. Most words were wrong and the output did not match the original sentence.

### Image 2: book1.jpg.jpeg

**Actual Urdu Text:**

شاعری ادب کی اعلیٰ ترین صنف ہے۔

**Tesseract Output:**

No text detected.

**What went wrong?**

Tesseract failed to detect the Urdu text and returned a blank output.

### Image 3: news1.jpg.jpeg

**Actual Urdu Text:**

غیر ملکی خبر رساں ادارے اے ایف پی کی رپورٹ کے

**Tesseract Output:**

غیرملکی خبررساں ادارے اے ایف پی کی رپورٹ کے

**What went wrong?**

Tesseract recognized most of the sentence but merged some words together and spacing was incorrect.

### Image 4: news11.jpg.jpeg

**Actual Urdu text**

کراچی (نیوز ڈیسک) پاکستان نے اپنی سب سے بڑی فعال چینی

**Tesseract Output:**

9:6

**What went wrong?**

Tesseract detected only a few incorrect characters and failed to recognize the Urdu sentence.

### Image 5: news10.jpg.jpeg

**Actual Urdu Text:**

سرپرستی میں منعقدہ ایک ثقافتی تقریب کے دوران

**Tesseract Output:**

وپ اہ اک ار ری کے لوان

**What went wrong?**

Tesseract recognized only a few incorrect words and missed most of the original text.

# **Summary**

Tesseract fails on Urdu because Urdu is a cursive script with connected characters and complex shapes. The default Tesseract OCR model could recognize only some parts of the text, while many words were incorrect, missing, or not detected. Therefore, a better OCR model trained specifically for Urdu is needed for accurate text recognition.